# $\omega=4\pi$: particle-Euler DTB versus direct neural-map parameter updates

This notebook compares two ways to evolve the same residual MMNN representation

$$
X_\theta(z)=T_\theta(z)
$$

for the two-player non-potential game

$$
b_\omega(x)=
\begin{pmatrix}
-\kappa x_1+A\sin(\omega x_2)\\
-\kappa x_2-A\sin(\omega x_1)
\end{pmatrix},
\qquad \omega=4\pi.
$$

Both methods use the same initial particles, identity-initialized MMNN, time grid, random tangent-coordinate subsets, truncated-SVD tolerance, and refined RK4 reference.

**Framework 1 — particle-Euler DTB.** The packaged DTB runner projects the physical velocity and advances the particle state and selected parameters with the same first-order tangent increment:

$$
\alpha_k=\arg\min_\alpha\|J_{S_k}(\theta_k,z)\alpha-b_\omega(X_k)\|_2^2,
$$

$$
X_{k+1}=X_k+hJ_{S_k}(\theta_k,z)\alpha_k,
\qquad
\theta_{k+1}[S_k]=\theta_k[S_k]+h\alpha_k.
$$

**Framework 2 — direct neural-map parameter update.** The particles are always defined by the current neural map. After the same least-squares projection, only the parameter state is stepped and the next particles are recomputed nonlinearly:

$$
\theta_{k+1}[S_k]=\theta_k[S_k]+h\alpha_k,
\qquad
X_{k+1}=T_{\theta_{k+1}}(z).
$$

The methods agree to first order in $h$ but differ by the nonlinear parameter-update remainder.

In [ ]:
from pathlib import Path
import copy
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Locate DTB_Ver3 locally/on PACE. Clone the branch only in Colab when absent.
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((p for p in candidates if (p / 'DTB_Ver3').is_dir()), None)
if repo_root is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run inside the repository or upload the DTB_Ver3 folder.')
    repo_root = Path('/content/dtb-colab-experiments')
    if not repo_root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo_root),
        ], check=True)
sys.path.insert(0, str(repo_root.resolve()))

from DTB_Ver3 import (
    ExperimentConfig,
    OscillatoryNonpotentialGame,
    ResidualMMNN,
    evaluate_dtb_projection,
    rk4_flow,
    run_experiment,
)
from DTB_Ver3.dtb import evaluate_model, flat_parameters
from DTB_Ver3.utils import (
    make_time_grid,
    paired_rms,
    resolve_device,
    resolve_dtype,
    snapshot_indices,
    to_numpy,
    warmup_cuda,
    write_csv,
    write_json,
)

output_dir = repo_root / 'DTB_Ver3' / 'results' / 'oscillatory_4pi_dtb_vs_direct_parameter_map'
output_dir.mkdir(parents=True, exist_ok=True)
print('repository:', repo_root)
print('output:', output_dir)

## 1. Shared controls

The comparison uses fixed initial labels. Therefore, every tangent matrix is

$$
J_{S_k}(\theta_k,z)=D_{\theta_{S_k}}T_{\theta_k}(z).
$$

A new random coordinate subset $S_k$ is selected at every step by the package DTB run. The direct neural-map method then reuses exactly those stored indices.

In [ ]:
SEED = 2026
KAPPA = 1.0
AMPLITUDE = 1.0
OMEGA = 4.0 * np.pi

N_PARTICLES = 10_000
STEP_SIZE = 0.001
FINAL_TIME = 0.2
SNAPSHOT_TIMES = (0.0, 0.05, 0.10, 0.15, 0.20)
RK4_REFERENCE_STEP = 0.000125

MMNN_WIDTH = 12
MMNN_RANK = 12
MMNN_DEPTH = 3
ACTIVATION = 'tanh'
TANGENT_SUBSET_SIZE = 128
SVD_RTOL = 1e-8
JACOBIAN_CHUNK = 512

DTYPE_NAME = 'float64'
DEVICE_NAME = 'auto'
PROGRESS_REPORTS = 10

if not np.isclose(FINAL_TIME / STEP_SIZE, round(FINAL_TIME / STEP_SIZE)):
    raise ValueError('FINAL_TIME must be an integer multiple of STEP_SIZE.')

device = resolve_device(DEVICE_NAME)
dtype = resolve_dtype(DTYPE_NAME)
warmup_cuda(device, dtype)
game = OscillatoryNonpotentialGame(KAPPA, AMPLITUDE, OMEGA)

torch.manual_seed(SEED + 100)
base_model = ResidualMMNN(
    2,
    width=MMNN_WIDTH,
    rank=MMNN_RANK,
    depth=MMNN_DEPTH,
    activation=ACTIVATION,
    dtype=dtype,
    zero_init_output=True,
).to(device)
dtb_model = copy.deepcopy(base_model)
direct_map_model = copy.deepcopy(base_model)

print({
    'omega_over_pi': OMEGA / np.pi,
    'particles': N_PARTICLES,
    'step_size': STEP_SIZE,
    'final_time': FINAL_TIME,
    'MMNN_width_rank_depth': (MMNN_WIDTH, MMNN_RANK, MMNN_DEPTH),
    'tangent_subset_size': TANGENT_SUBSET_SIZE,
    'device': str(device),
    'dtype': str(dtype),
})

## 2. Framework 1: packaged particle-Euler DTB

This block delegates particle evolution, tangent construction, random subset selection, SVD projection, parameter updates, RK4 integration, snapshots, and diagnostics to `run_experiment` from the PACE bundle.

In [ ]:
dtb_config = ExperimentConfig(
    dynamics='deterministic',
    run_reference=True,
    reference_integrator='rk4',
    reference_step_size=RK4_REFERENCE_STEP,
    particle_count=N_PARTICLES,
    initial_law='uniform',
    initial_low=-1.0,
    initial_high=1.0,
    step_size=STEP_SIZE,
    final_time=FINAL_TIME,
    snapshot_times=SNAPSHOT_TIMES,
    width=MMNN_WIDTH,
    rank=MMNN_RANK,
    depth=MMNN_DEPTH,
    activation=ACTIVATION,
    model_kind='residual_mmnn',
    zero_init_output=True,
    basis_size=TANGENT_SUBSET_SIZE,
    subset_tangent_selection='resample_each_step',
    tangent_input_mode='fixed_initial_labels',
    track_network_map=False,
    svd_rtol=SVD_RTOL,
    jacobian_chunk=JACOBIAN_CHUNK,
    seed=SEED,
    dtype=DTYPE_NAME,
    device=DEVICE_NAME,
    progress_reports=PROGRESS_REPORTS,
    save_outputs=False,
)
dtb = run_experiment(game, dtb_config, model=dtb_model)
if dtb.reference_final_particles is None:
    raise RuntimeError('RK4 reference was not returned.')
print({
    'particle_Euler_DTB_final_RMS': dtb.final_paired_rms,
    'final_relative_projection_error': dtb.final_relative_projection_error,
    'final_alpha_norm': dtb.final_alpha_norm,
})

## 3. Framework 2: direct neural-map parameter update

The loop below uses package functions for functional MMNN evaluation, tangent construction, SVD projection, and RK4. It does not maintain a separately accumulated particle state. At every step it evaluates $X_k=T_{\theta_k}(z)$, projects $b_\omega(X_k)$, updates the selected parameter coordinates, and evaluates the updated map again.

In [ ]:
labels = torch.as_tensor(dtb.initial_particles, dtype=dtype, device=device)
theta = torch.as_tensor(dtb.initial_parameters, dtype=dtype, device=device)
initial_theta, structure = flat_parameters(direct_map_model)
torch.testing.assert_close(theta, initial_theta, rtol=0.0, atol=0.0)
network_particles = evaluate_model(theta, labels, direct_map_model, structure).detach()
torch.testing.assert_close(network_particles, labels, rtol=0.0, atol=0.0)

# Reuse the package-generated time grid, reference method, and subset schedule.
times = dtb.times.copy()
projection_times = times[:-1].copy()
snapshot_steps = snapshot_indices(times, SNAPSHOT_TIMES)
reference_particles = labels.detach().clone()
direct_map_snapshots = {float(times[0]): to_numpy(network_particles).copy()}
direct_reference_snapshots = {float(times[0]): to_numpy(reference_particles).copy()}

direct_relative_projection_error = []
direct_alpha_norm = []
direct_trajectory_rms = [0.0]
direct_condition = []
direct_rank = []

for step in range(len(times) - 1):
    current_time = float(times[step])
    h = float(times[step + 1] - times[step])
    selected = torch.as_tensor(
        dtb.selected_indices_history[step],
        dtype=torch.long,
        device=device,
    )

    # X_k is the current neural solution map evaluated at immutable labels z.
    network_particles = evaluate_model(
        theta, labels, direct_map_model, structure
    ).detach()
    target_velocity = game.velocity(network_particles, current_time)
    projection = evaluate_dtb_projection(
        theta,
        selected,
        labels,
        target_velocity,
        direct_map_model,
        structure,
        chunk_size=JACOBIAN_CHUNK,
        svd_rtol=SVD_RTOL,
    )

    # Direct parameter Euler update, followed by a nonlinear map evaluation.
    next_theta = theta.detach().clone()
    next_theta[selected] += h * projection.alpha.detach()
    if not torch.isfinite(next_theta).all():
        raise FloatingPointError('direct parameter update produced nonfinite parameters')
    theta = next_theta
    network_particles = evaluate_model(
        theta, labels, direct_map_model, structure
    ).detach()

    reference_particles = rk4_flow(
        game,
        reference_particles,
        current_time,
        h,
        maximum_step=RK4_REFERENCE_STEP,
    )
    direct_relative_projection_error.append(float(projection.relative_residual.item()))
    direct_alpha_norm.append(float(torch.linalg.vector_norm(projection.alpha).item()))
    direct_condition.append(projection.condition_number)
    direct_rank.append(projection.retained_rank)
    direct_trajectory_rms.append(paired_rms(network_particles, reference_particles))

    state_index = step + 1
    if state_index in snapshot_steps:
        snapshot_time = snapshot_steps[state_index]
        direct_map_snapshots[snapshot_time] = to_numpy(network_particles).copy()
        direct_reference_snapshots[snapshot_time] = to_numpy(reference_particles).copy()
    if state_index % max(1, (len(times) - 1) // PROGRESS_REPORTS) == 0:
        print(
            f'{state_index}/{len(times)-1} | t={times[state_index]:g} | '
            f'projection={direct_relative_projection_error[-1]:.3e} | '
            f'alpha={direct_alpha_norm[-1]:.3e} | '
            f'RMS={direct_trajectory_rms[-1]:.3e}',
            flush=True,
        )

# At the first step the states are identical, so both projections must agree.
np.testing.assert_allclose(direct_alpha_norm[0], dtb.alpha_norm[0], rtol=1e-12, atol=1e-12)
np.testing.assert_allclose(
    direct_relative_projection_error[0],
    dtb.relative_projection_error[0],
    rtol=1e-12,
    atol=1e-12,
)

# The independently integrated RK4 trajectories must agree exactly.
np.testing.assert_array_equal(to_numpy(reference_particles), dtb.reference_final_particles)
print({
    'direct_parameter_map_final_RMS': direct_trajectory_rms[-1],
    'final_relative_projection_error': direct_relative_projection_error[-1],
    'final_alpha_norm': direct_alpha_norm[-1],
})

## 4. Snapshot comparison with RK4

Each column is one common time. The three rows show particle-Euler DTB, the direct neural-map parameter update, and refined RK4. Within each column all rows use the same particle indices and axis limits.

In [ ]:
PLOT_MAX_POINTS = 10_000
snapshot_times = tuple(float(value) for value in SNAPSHOT_TIMES)
fig, axes = plt.subplots(
    3,
    len(snapshot_times),
    figsize=(3.2 * len(snapshot_times), 9.2),
    squeeze=False,
    constrained_layout=True,
)
row_names = ('particle-Euler DTB', 'direct parameter map', 'RK4 reference')
row_colors = ('tab:blue', 'tab:orange', 'tab:green')

for column_index, snapshot_time in enumerate(snapshot_times):
    dtb_cloud = dtb.dtb_snapshots[snapshot_time]
    direct_cloud = direct_map_snapshots[snapshot_time]
    reference_cloud = dtb.reference_snapshots[snapshot_time]
    clouds = (dtb_cloud, direct_cloud, reference_cloud)
    particle_count = reference_cloud.shape[0]
    if particle_count <= PLOT_MAX_POINTS:
        plot_indices = np.arange(particle_count)
    else:
        plot_rng = np.random.default_rng(SEED + column_index)
        plot_indices = np.sort(
            plot_rng.choice(particle_count, PLOT_MAX_POINTS, replace=False)
        )

    combined = np.concatenate(clouds, axis=0)
    lower = combined.min(axis=0)
    upper = combined.max(axis=0)
    span = np.maximum(upper - lower, 1e-12)
    padding = 0.05 * span

    for row_index, (cloud, name, color) in enumerate(zip(clouds, row_names, row_colors)):
        axis = axes[row_index, column_index]
        shown = cloud[plot_indices]
        axis.scatter(
            shown[:, 0], shown[:, 1],
            s=3, alpha=0.28, color=color, linewidths=0, rasterized=True,
        )
        axis.set_xlim(lower[0] - padding[0], upper[0] + padding[0])
        axis.set_ylim(lower[1] - padding[1], upper[1] + padding[1])
        axis.set_aspect('equal', adjustable='box')
        axis.grid(True, alpha=0.18)
        axis.set_xlabel(r'$x_1$')
        axis.set_ylabel(r'$x_2$')
        axis.set_title(f'{name}\n$t={snapshot_time:g}$', fontsize=10)

fig.suptitle(r'$\omega=4\pi$: matched particle snapshots', fontsize=15)
snapshot_plot = output_dir / 'snapshots_dtb_direct_map_rk4.png'
fig.savefig(snapshot_plot, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', snapshot_plot)

## 5. Projection and coefficient trajectories

The relative projection error is

$$
r_k=\frac{\|J_{S_k}\alpha_k-b_\omega(X_k)\|_2}{\|b_\omega(X_k)\|_2},
$$

and the coefficient diagnostic is

$$
a_k=\|\alpha_k\|_2.
$$

A small $r_k$ with a very large $a_k$ indicates that the fit relies on weak tangent directions. The third panel shows how these local differences accumulate into trajectory error against RK4.

In [ ]:
dtb_alpha = np.asarray(dtb.alpha_norm, dtype=float)
dtb_relative = np.asarray(dtb.relative_projection_error, dtype=float)
dtb_rms = np.asarray(dtb.trajectory_rms_error, dtype=float)
direct_alpha = np.asarray(direct_alpha_norm, dtype=float)
direct_relative = np.asarray(direct_relative_projection_error, dtype=float)
direct_rms = np.asarray(direct_trajectory_rms, dtype=float)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
axes[0].plot(projection_times, dtb_alpha, label='particle-Euler DTB', color='tab:blue')
axes[0].plot(projection_times, direct_alpha, label='direct parameter map', color='tab:orange')
axes[0].set_yscale('log')
axes[0].set(xlabel='time', ylabel=r'$\|\alpha_k\|_2$', title='coefficient norm')

axes[1].plot(projection_times, dtb_relative, label='particle-Euler DTB', color='tab:blue')
axes[1].plot(projection_times, direct_relative, label='direct parameter map', color='tab:orange')
axes[1].set_yscale('log')
axes[1].set(xlabel='time', ylabel='relative projection error', title='tangent projection error')

axes[2].plot(times, dtb_rms, label='particle-Euler DTB', color='tab:blue')
axes[2].plot(times, direct_rms, label='direct parameter map', color='tab:orange')
axes[2].set_yscale('log')
axes[2].set(xlabel='time', ylabel='paired RMS against RK4', title='trajectory error')

for axis in axes:
    axis.grid(True, which='both', alpha=0.3)
    axis.legend()
diagnostics_plot = output_dir / 'alpha_projection_and_rk4_rms.png'
fig.savefig(diagnostics_plot, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', diagnostics_plot)

## 6. Save reproducible arrays

The CSV stores one row per projection time for both methods. The NPZ stores every plotted snapshot, time history, and final parameter vector.

In [ ]:
rows = []
for method, alpha_values, relative_values, rms_values, conditions, ranks in (
    (
        'particle_euler_dtb', dtb_alpha, dtb_relative, dtb_rms[1:],
        np.asarray(dtb.jacobian_condition), np.asarray(dtb.retained_rank),
    ),
    (
        'direct_parameter_map', direct_alpha, direct_relative, direct_rms[1:],
        np.asarray(direct_condition), np.asarray(direct_rank),
    ),
):
    for index, time_value in enumerate(projection_times):
        rows.append((
            method, float(time_value), float(alpha_values[index]),
            float(relative_values[index]), float(rms_values[index]),
            float(conditions[index]), int(ranks[index]),
        ))

diagnostics_csv = write_csv(
    output_dir / 'trajectory_diagnostics.csv',
    (
        'method', 'time', 'alpha_norm', 'relative_projection_error',
        'paired_rms_against_rk4', 'condition_number', 'retained_rank',
    ),
    rows,
)
npz_path = output_dir / 'comparison_arrays.npz'
np.savez_compressed(
    npz_path,
    times=times,
    projection_times=projection_times,
    dtb_alpha_norm=dtb_alpha,
    direct_alpha_norm=direct_alpha,
    dtb_relative_projection_error=dtb_relative,
    direct_relative_projection_error=direct_relative,
    dtb_trajectory_rms=dtb_rms,
    direct_trajectory_rms=direct_rms,
    snapshot_times=np.asarray(snapshot_times),
    dtb_snapshots=np.stack([dtb.dtb_snapshots[t] for t in snapshot_times]),
    direct_map_snapshots=np.stack([direct_map_snapshots[t] for t in snapshot_times]),
    rk4_snapshots=np.stack([dtb.reference_snapshots[t] for t in snapshot_times]),
    dtb_final_parameters=dtb.final_parameters,
    direct_map_final_parameters=to_numpy(theta),
    selected_indices_history=dtb.selected_indices_history,
)
write_json(output_dir / 'configuration.json', {
    'seed': SEED,
    'kappa': KAPPA,
    'amplitude': AMPLITUDE,
    'omega': OMEGA,
    'omega_over_pi': OMEGA / np.pi,
    'particles': N_PARTICLES,
    'step_size': STEP_SIZE,
    'final_time': FINAL_TIME,
    'snapshot_times': snapshot_times,
    'rk4_reference_step': RK4_REFERENCE_STEP,
    'MMNN': {
        'width': MMNN_WIDTH,
        'rank': MMNN_RANK,
        'depth': MMNN_DEPTH,
        'activation': ACTIVATION,
    },
    'tangent_subset_size': TANGENT_SUBSET_SIZE,
    'subset_selection': 'resample_each_step_matched_between_methods',
    'svd_rtol': SVD_RTOL,
    'jacobian_chunk': JACOBIAN_CHUNK,
    'dtype': DTYPE_NAME,
    'device': str(device),
})
print('saved:', diagnostics_csv)
print('saved:', npz_path)

## 7. Interpretation guide

- If the two methods agree as $h$ decreases, their discrepancy at the current step size is primarily the nonlinear remainder produced by evaluating $T_{\theta_k+h\alpha_k}$.
- If particle-Euler DTB has smaller projection error, its separately accumulated particles are producing a target velocity better aligned with the current tangent basis.
- If the direct parameter map has smaller RK4 RMS, repeatedly enforcing $X_k=T_{\theta_k}(z)$ is reducing drift between the physical state and the learned solution map.
- Large coefficient norms or condition numbers indicate reliance on weak singular directions even when the projection residual is small.

In [ ]:
archive_path = Path(shutil.make_archive(
    str(output_dir),
    'zip',
    root_dir=output_dir,
))
print('result archive:', archive_path)